# Deep Learning 008 — Notation and Counting Parameters

Notation is only worth a notebook if it lets you compute something. It does: **the
number of parameters in a dense network**, which is the number you will be asked for in
every interview and the one that decides whether a model fits on your hardware.

In [ ]:
import numpy as np

def count_params(layer_dims, bias=True):
    '''layer_dims = [inputs, hidden..., outputs]. Returns per-layer and total.'''
    per = []
    for i in range(len(layer_dims) - 1):
        n_in, n_out = layer_dims[i], layer_dims[i + 1]
        w = n_in * n_out
        b = n_out if bias else 0
        per.append((f'{n_in} -> {n_out}', w, b, w + b))
    return per, sum(p[3] for p in per)

def report(layer_dims, name=''):
    per, total = count_params(layer_dims)
    print(f'{name}  {layer_dims}')
    print(f"  {'layer':>12}{'weights':>10}{'biases':>8}{'total':>9}")
    for lab, w, b, t in per:
        print(f'  {lab:>12}{w:>10,}{b:>8,}{t:>9,}')
    print(f"  {'TOTAL':>12}{'':>10}{'':>8}{total:>9,}\n")
    return total

report([4, 3, 2, 1], 'the lesson network')
report([2, 8, 1], 'XOR-sized')
report([784, 128, 10], 'MNIST, one hidden layer')

The formula is worth saying out loud: **a dense layer from $n$ inputs to $m$ outputs has
$nm$ weights and $m$ biases.** Everything else is bookkeeping.

## Where the parameters actually live

Count MNIST and look at the split. This is the pattern that recurs through the entire
track.

In [ ]:
per, total = count_params([784, 128, 10])
print(f'{"layer":>12}{"params":>10}{"share":>9}')
for lab, w, b, t in per:
    print(f'{lab:>12}{t:>10,}{t/total:>9.1%}')
print(f'\nThe first layer is {per[0][3]/total:.1%} of the model - and its size is set')
print('by the INPUT dimension (784 pixels), not by anything interesting.')

In [ ]:
# Widening the hidden layer: cost grows linearly, and the input layer dominates.
print(f'{"hidden":>8}{"total":>12}{"first layer share":>20}')
for h in (16, 64, 128, 512, 2048):
    per, total = count_params([784, h, 10])
    print(f'{h:>8}{total:>12,}{per[0][3]/total:>20.1%}')

## Deeper vs wider, at a fixed budget

A useful exercise: two networks with roughly the same parameter count, arranged
differently.

In [ ]:
for dims in ([784, 300, 10], [784, 256, 64, 10], [784, 128, 128, 64, 32, 10]):
    _, total = count_params(dims)
    print(f'{str(dims):<36} {total:>9,} parameters, {len(dims)-1} layers')

> **TensorFlow is optional here.** Every cell above runs on NumPy alone. The Keras
> comparison below is the check that your hand count is right — if TensorFlow is not
> installed the cell skips itself with a message instead of failing.

In [ ]:
try:
    from tensorflow import keras
    for dims in ([4, 3, 2, 1], [784, 128, 10]):
        m = keras.Sequential(
            [keras.layers.Input(shape=(dims[0],))] +
            [keras.layers.Dense(n, activation='relu') for n in dims[1:-1]] +
            [keras.layers.Dense(dims[-1], activation='sigmoid')])
        ours = count_params(dims)[1]
        print(f'{dims}: ours {ours:,}  keras {m.count_params():,}  '
              f'match={ours == m.count_params()}')
except ImportError:
    print('TensorFlow not installed - skipping the Keras cross-check.')
    print('The formula n*m + m is not in doubt; run this cell later if you want')
    print('to see model.summary() agree with it line by line.')

## Exercises

1. Add a `bias=False` variant to the table. How much do biases actually cost, as a
   percentage?
2. A network has 1,000,000 parameters, 784 inputs and 10 outputs, with **one** hidden
   layer. Solve for the hidden width, then check it with `count_params`.
3. Extend `count_params` to handle a convolutional layer
   (`kernel_h * kernel_w * in_channels * out_channels + out_channels`) and count
   LeNet-5. Compare its convolutional total against its dense total — the answer is the
   whole point of lesson 049 onward.